# Redrob Candidate Ranker — Sandbox Demo

This notebook is the **sandbox link** required by the Redrob hackathon submission spec (Section 10.5). It runs the full ranking pipeline end-to-end on a small sample of candidates so the architecture can be verified without needing the full 100K-candidate pool or a local environment.

Full code, the complete rubric, and the real 100K-scale output live in the GitHub repo:
**https://github.com/srivathsan1510/redrob-india-runs-ranker**

This notebook clones that repo and runs `precompute.py` + `rank.py` on the 50-candidate sample file included in the repo (`data/sample_candidates.json`), which mirrors exactly what the real pipeline does on the full pool — just at a size that's fast and easy to inspect interactively.

## 1. Clone the repo and install dependencies

In [ ]:
!git clone https://github.com/srivathsan1510/redrob-india-runs-ranker.git
%cd redrob-india-runs-ranker
!pip install -q -r requirements.txt

## 2. Phase A — Precompute (features + honeypot checks + embeddings)

On the real 100K-candidate pool this step has no time limit and is run once, locally. Here it runs against the included 50-candidate sample, which takes well under a minute including the one-time sentence-transformer model download.

In [ ]:
!python precompute.py \
  --candidates data/sample_candidates.json \
  --jd-text-file config/jd_text.txt \
  --out-dir artifacts_demo \
  --embedder sentence-transformers

## 3. Phase B — Rank (the timed step: 5 min / 16GB / CPU / no network on the real 100K pool)

This is pure vectorized arithmetic over the artifacts from step 2 — no embedding computation happens here, which is why it stays fast and network-free regardless of pool size.

In [ ]:
!python rank.py \
  --candidates data/sample_candidates.json \
  --artifacts-dir artifacts_demo \
  --out outputs/demo_submission.csv \
  --top-n 50

## 4. Inspect the ranked output

Top of the list should show genuinely relevant titles (ML/search/ranking engineers); irrelevant titles (e.g. Accountant, Graphic Designer, Marketing Manager — present in the sample pool as distractors) should sink toward the bottom.

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 150)

df = pd.read_csv('outputs/demo_submission.csv')
print(f"Ranked {len(df)} candidates.\n")
print("=== Top 10 ===")
display(df.head(10))

In [ ]:
print("=== Bottom 10 ===")
display(df.tail(10))

## 5. Cross-reference rank against actual job titles

This confirms the ranker is genuinely separating relevant from irrelevant candidates, not just reproducing the input order.

In [ ]:
import json

candidates = json.load(open('data/sample_candidates.json'))
title_lookup = {c['candidate_id']: c['profile']['current_title'] for c in candidates}

df['current_title'] = df['candidate_id'].map(title_lookup)
display(df[['rank', 'candidate_id', 'current_title', 'score']])

## 6. Validate against the official format checker

In [ ]:
# Note: the official validator expects exactly 100 rows (the real submission
# requirement). This demo only ranks the 50 sample candidates, so the row-count
# check will correctly report a mismatch here — that's expected for a sandbox
# demo and is NOT present in the real outputs/submission.csv in this repo,
# which was produced from the full 100K pool and validates cleanly.
!python validate_submission.py outputs/demo_submission.csv

## About the approach

Two-phase pipeline (precompute / rank) with a relevance-first, gated scoring formula: semantic JD-fit (sentence-transformer embeddings) and rule-based skill-cluster matching form the core relevance score, which experience-band and location fit can only *modulate*, not replace — so a candidate with strong experience but zero relevant skills can't outrank a genuine match. Skill-cluster matches grounded in actual career-history text count far more than matches found only in a self-reported skills list, which directly defeats keyword-stuffing. Multiplicative penalties apply for six rule-based disqualifiers, narrative-consistency violations, and honeypot-consistency checks. Reasoning text is template-based and fact-grounded — every claim traces to an actual field in the candidate's record — with tone calibrated to both rank position and penalty severity.

Full rubric: `config/jd_profile.yaml` in the repo. Full methodology: `submission_metadata.yaml`.